# Checkpoint V · Examen blanc — notebook de l'examen

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/checkpoints/partie_5/02_examen_notebook.ipynb)

Ce notebook accompagne le sujet `01_examen_sujet.md`.

- **Partie A, pendant l'examen** : les deux questions de code, CP5.5 (🔨, 1,5 point, avec NumPy seulement) et CP5.6 (🔨, 2 points, avec PyTorch). Écris ton code dans les cellules à compléter, sans `mylearn`, sans les corrigés et sans assistant IA ; `help()` et les documentations officielles de NumPy et de PyTorch sont permis. Rien n'est vérifié pendant l'examen : les cellules « essai » montrent seulement ce que fait ton code. Tes réponses chiffrées et tes explications vont sur ta feuille (`04_mes_reponses.md`).
- **Partie B, après l'examen** : la vérification automatique de ton code et des réponses de ta feuille. Ses cellules ne vérifient rien tant que `EXAM_OVER` vaut `False`.

« Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳ (partie A) ou ⏸️ (partie B).

> Travaille dans **ta copie** (`mon_travail/checkpoints/partie_5/02_examen_notebook.ipynb`, créée par `python tools/start_chapter.py CP5`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

## Partie A · Pendant l'examen : les questions de code

In [ ]:
# Data and tools of the code questions CP5.5 and CP5.6 (run this cell first)
import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# CP5.5: a mini-batch of 3 sequences of 5 steps with 2 features, and the parameters of a simple RNN of 4 neurons
rng = np.random.default_rng(55)
X_S = rng.normal(size=(3, 5, 2))                       # (N, T, D): batch, time, features
H0_S = np.zeros((3, 4))                                # (N, H): the starting state
WX_S = rng.normal(0.0, 0.8, size=(2, 4))               # (D, H)
WH_S = rng.normal(0.0, 0.8, size=(4, 4))               # (H, H)
B_S = rng.normal(0.0, 0.3, size=4)                     # (H,)


def rnn_cell(x_t, h, Wx, Wh, b):
    """One step of a simple RNN for a mini-batch: x_t (N, D), h (N, H) -> tanh(x_t @ Wx + h @ Wh + b), (N, H)."""
    return np.tanh(x_t @ Wx + h @ Wh + b)


# CP5.6: 3 072 images of Fashion-MNIST in [0, 1], float64 (2 048 to train, 1 024 to validate)
_X6, _Y6 = wb.datasets.load_fashion_mnist("train", n=3072, normalize=True, seed=56)
X_IMG = torch.from_numpy(_X6.astype(np.float64)).unsqueeze(1)      # (3072, 1, 28, 28)
Y_IMG = torch.from_numpy(_Y6.astype(np.int64))                     # (3072,)


def make_cnn(seed=56):
    """A new small CNN with dropout, in float64 (the same starting weights for the same seed)."""
    torch.manual_seed(seed)
    return nn.Sequential(
        nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),       # (N, 8, 14, 14)
        nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),      # (N, 16, 7, 7)
        nn.Flatten(), nn.Dropout(0.25), nn.Linear(16 * 7 * 7, 10)).double()


def make_loaders(seed=56, n_train=2048, n_val=1024):
    """(train_loader, val_loader): mini-batches of 64 shuffled images, then of 300 images (the last one is smaller)."""
    train = TensorDataset(X_IMG[:n_train], Y_IMG[:n_train])
    val = TensorDataset(X_IMG[2048:2048 + n_val], Y_IMG[2048:2048 + n_val])
    return (DataLoader(train, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(seed)),
            DataLoader(val, batch_size=300))


CHECKPOINT = Path(tempfile.gettempdir()) / "cp5_6_best.pt"     # where the try of CP5.6 saves the best weights
_ORIGINALS = {"X_S": X_S.copy(), "H0_S": H0_S.copy()}          # part B restores them if a try changed them
print("X_S:", X_S.shape, "· images:", tuple(X_IMG.shape), "· labels:", tuple(Y_IMG.shape),
      "· CNN parameters:", sum(p.numel() for p in make_cnn().parameters()))

### CP5.5 — Écrire `rnn_forward` à partir d'une cellule fournie (notebook, NumPy) 🔨 ★ ⏱️ 10 min · 1,5 point
**Objectif :** dérouler une couche récurrente simple sur toute une séquence, pour un mini-batch.

Avec NumPy seulement. `rnn_cell(x_t, h, Wx, Wh, b)` (fournie) fait un pas : $h_t = \tanh(x_t W_x + h_{t-1} W_h + b)$, avec `x_t` de forme `(N, D)` et `h` de forme `(N, H)`.

a) `rnn_forward(X, h0, Wx, Wh, b)` reçoit `X` de forme `(N, T, D)` (batch, temps, features : la convention `batch_first=True`) et l'état de départ `h0`, de forme `(N, H)`. Elle rend le couple `(H_all, h_last)` : `H_all`, de forme `(N, T, H)`, où `H_all[:, t]` est l'état **après** le pas `t` (`h0` n'y figure pas), et `h_last`, l'état après le dernier pas, de forme `(N, H)`. Elle ne modifie pas `X`. **[1,2]**
b) Avec les données de la cellule (`X_S`, `H0_S`, `WX_S`, `WH_S`, `B_S`), reporte sur ta feuille `H_all[1, 4, 0]` (4 décimales). **[0,3]**

In [ ]:
def rnn_forward(X, h0, Wx, Wh, b):
    """(H_all (N, T, H), h_last (N, H)): the states after each step of rnn_cell over X (N, T, D); CP5.5 a)."""
    raise NotImplementedError("rnn_forward")  # TODO CP5.5 a)

In [ ]:
# The tries of part A show what your code does (no check during the exam)
import sys
import traceback


def try_code(question, run):
    """Run one try; a bug shows its traceback without stopping « Run all » (part B must still run)."""
    with wb.attempt(question):
        try:
            run()
        except NotImplementedError:
            raise
        except Exception as exc:
            if isinstance(exc, TypeError) and "NoneType" in str(exc):
                raise                                       # wb.attempt: « as-tu oublié le return ? »
            traceback.print_exc(file=sys.stdout)
            print(f"❌ {question} : ton code lève l'erreur ci-dessus (la suite du notebook s'exécute quand même).")


def try_5():
    H_all, h_last = rnn_forward(X_S, H0_S, WX_S, WH_S, B_S)
    print("H_all:", np.shape(H_all), "· h_last:", np.shape(h_last))
    print("H_all[1, 4, 0] =", round(float(H_all[1, 4, 0]), 4))


try_code("CP5.5", try_5)

### CP5.6 — Entraîner avec early stopping et garder le meilleur modèle (notebook, PyTorch) 🔨 ★★ ⏱️ 20 min · 2 points
**Objectif :** écrire de mémoire une boucle d'entraînement PyTorch complète : modes entraînement et évaluation, loss de validation par exemple, sauvegarde des meilleurs poids, arrêt après `patience` epochs sans progrès.

Avec PyTorch. `make_cnn()` rend un petit CNN neuf (avec dropout, en `float64`) et `make_loaders()` le couple `(train_loader, val_loader)` sur 2 048 et 1 024 images de Fashion-MNIST ; les mini-batches de validation ont 300 images, sauf le dernier (124).

a) `fit_early_stopping(model, train_loader, val_loader, max_epochs, patience, path, lr=1e-3)` : **[1,6]**
- crée **un** optimiseur, `torch.optim.Adam(model.parameters(), lr=lr)` ;
- à chaque epoch (numérotées à partir de 1, au plus `max_epochs`) : en mode entraînement, un passage sur `train_loader` (cross-entropy moyenne du mini-batch sur les logits, `zero_grad`, `backward`, `step`) ; la loss d'entraînement de l'epoch est la moyenne **par exemple** des losses des mini-batches (chacune pesée par son nombre d'exemples) ; puis, en mode évaluation et sans gradient, la loss de validation : la cross-entropy moyenne **par exemple** sur tout `val_loader` ;
- si la loss de validation est **strictement** plus petite que la meilleure vue jusque-là, sauvegarde `model.state_dict()` dans `path` (`torch.save`) et retient l'epoch ; sinon, compte une epoch sans progrès ; l'entraînement s'arrête dès que `patience` epochs **de suite** n'ont pas progressé ;
- à la fin, recharge dans `model` les poids sauvegardés (`torch.load(path, weights_only=True)`) et rend le dictionnaire `{"train_loss": [...], "val_loss": [...], "best_epoch": k}` : une valeur (un `float`) par epoch faite, et l'epoch des poids gardés, numérotée à partir de 1.

b) La cellule d'essai entraîne `make_cnn()` sur `make_loaders()` avec `max_epochs=30`, `patience=3` et `lr=3e-3` (une dizaine de secondes) : reporte sur ta feuille $[\text{best\_epoch}, \text{loss de validation à cette epoch}]$ (4 décimales). **[0,4]**

In [ ]:
def fit_early_stopping(model, train_loader, val_loader, max_epochs, patience, path, lr=1e-3):
    """Adam, early stopping on the validation loss (mean per example), best state_dict saved in path and reloaded;
    returns {"train_loss": [...], "val_loss": [...], "best_epoch": k}; CP5.6 a)."""
    raise NotImplementedError("fit_early_stopping")  # TODO CP5.6 a)

In [ ]:
# Try fit_early_stopping (about ten seconds; no check during the exam)
def try_6():
    model_try = make_cnn()
    history_try = fit_early_stopping(model_try, *make_loaders(), 30, 3, CHECKPOINT, lr=3e-3)
    print("epochs done:", len(history_try["val_loss"]), "· best_epoch:", history_try["best_epoch"])
    print("validation losses:", [round(float(v), 4) for v in history_try["val_loss"]])


try_code("CP5.6", try_6)

---

### 🛑 Fin de la partie A

L'examen s'arrête ici. Rends ta copie (enregistre ce notebook et ta feuille), puis, seulement après, passe à la partie B.

## Partie B · Après l'examen : vérification automatique

**Ne commence cette partie qu'une fois ta copie terminée**, examen rendu. Elle vérifie le code de la partie A, puis les réponses que tu as écrites sur ta feuille, question par question : ✅ juste, ❌ faux (avec une piste), ⏳ pas rempli. Elle ne note pas : les démarches, les justifications et les questions rédigées se corrigent avec `03_examen_corrige.md`, qui donne aussi le barème.

1. Passe `EXAM_OVER` à `True` dans la cellule ci-dessous, puis exécute-la.
2. Exécute les vérifications du code (CP5.5 et CP5.6 ; la seconde entraîne le CNN plusieurs fois, une trentaine de secondes).
3. Reporte ensuite chaque réponse de ta feuille : **la valeur** que tu as écrite, pas un nouveau calcul (sinon tu ne vérifies plus ta copie). En Python, le séparateur décimal est un **point** (`0.125`), une matrice s'écrit comme une liste de lignes (`[[1, 2], [3, 4]]`), une forme comme une chaîne (`"(2, 5, 7, 7)"`) et les lettres entre guillemets (`"XYZ"`).

Si tu as fermé le notebook depuis l'examen, exécute d'abord la cellule de configuration (tout en haut) et les cellules de la partie A, sauf les essais.

In [ ]:
EXAM_OVER = False   # set it to True once your exam is finished, then run the cells of part B

import re

from wb import exams
from wb.nbtools import StopCheck, as_numpy, letters_text, pair, shape_text, verdict

RESULTS = {}        # sub-ID -> result of wb.check
PROPERTIES = {}     # question -> [(property, passed)], reset each time its check cell runs
PENDING = set()     # the code questions not written yet (⏳)
_REMINDED = []      # the full reminder is printed once, then a short one


def exam_over():
    """True once the exam is finished; otherwise print a reminder (and the cell checks nothing)."""
    if not EXAM_OVER:
        print("⏸️ Rien n'est vérifié (EXAM_OVER vaut False)." if _REMINDED else
              "⏸️ Vérification après l'examen : passe EXAM_OVER à True dans la première cellule de la partie B.")
        _REMINDED.append(True)
        return False
    if "wb" not in globals():
        print("⚠️ Exécute d'abord la cellule de configuration, tout en haut du notebook.")
        return False
    return True


def ready(question, *names):
    """True when the cells of part A that define `names` have run (after a kernel restart, run them again)."""
    missing = [name for name in names if name not in globals()]
    if missing:
        verb = "n'existe" if len(missing) == 1 else "n'existent"
        print(f"⚠️ {question} : exécute d'abord les cellules de la partie A ({', '.join(missing)} {verb} pas encore).")
    return not missing


def prop(question, ok, success, failure):
    """One property of the code of part A: ✅ or ❌ (wb.nbtools.verdict), remembered for the summary."""
    PROPERTIES.setdefault(question, []).append((success or failure, bool(ok)))
    verdict(question, ok, success, failure)
    return bool(ok)


def safe(function, *args, **kwargs):
    """(True, result) or (False, the text of the error), or (None, None) while the code is not written:
    a bug of the exam code must not stop part B."""
    try:
        return True, function(*args, **kwargs)
    except NotImplementedError:
        return None, None
    except Exception as exc:
        return False, f"{type(exc).__name__}: {str(exc)[:300]}"


_LIBRARY_ADVICE = " ; si elle fait partie de `mylearn`, ses tests (`pytest`) disent quel cas pose problème."


def code_check(ex_id, value, reminder=""):
    """wb.check of a value computed by your exam function (no advice about mylearn: it is not used here)."""
    result = wb.check(ex_id, value, computed=True, quiet=True)
    message = result.message.replace(_LIBRARY_ADVICE, ".")
    if not result and reminder and not message.startswith("Erreur classique"):
        message += f" Rappel : {reminder}"
    print(f"{'✅' if result else '❌'} Ex {ex_id} : {message}")
    RESULTS[ex_id] = result
    return result


def as_outcome(value):
    """CP5.1: a shape typed as "(8, 16)", "[8, 16]", "8 x 16" or a tuple -> "(8, 16)"; « erreur » for an error."""
    if isinstance(value, str) and re.search(r"(?i)err|exception|runtime|plante", value):
        return "erreur"
    return shape_text(value)


def sorted_letters(value):
    """A set of letters typed in any order ("K, A et E") -> its letters in alphabetical order ("AEK")."""
    return letters_text(value, sort=True)


def restore_data(question, names):
    """Put back the data of part A if a try of part A changed them in place (a function that writes into its input)."""
    changed = [name for name in names
               if name not in globals() or not np.array_equal(globals()[name], _ORIGINALS[name])]
    for name in changed:
        globals()[name] = _ORIGINALS[name].copy()
    if changed:
        print(f"⚠️ {question} : {', '.join(changed)} avai{'en' if len(changed) > 1 else ''}t changé depuis la cellule des "
              f"données (une fonction de la partie A écrit-elle dans ce qu'elle reçoit ?) : la vérification repart "
              f"des données d'origine.")

### Le code de la partie A

In [ ]:
# Checks of rnn_forward (CP5.5), against nn.RNN
if exam_over() and ready("CP5.5", "np", "torch", "nn", "X_S", "H0_S", "WX_S", "WH_S", "B_S", "rnn_cell", "_ORIGINALS",
                         "rnn_forward"):
    PROPERTIES["CP5.5"] = []
    PENDING.discard("CP5.5")
    RESULTS.pop("CP5.5b", None)
    restore_data("CP5.5", ["X_S", "H0_S"])
    oracle_5 = exams.cp5_rnn                            # (output, h_n) of nn.RNN(batch_first=True), the same weights

    X_in = X_S.copy()
    ok, result = safe(rnn_forward, X_in, H0_S.copy(), WX_S, WH_S, B_S)
    if ok is None:
        print("⏳ Ex CP5.5 : rnn_forward n'est pas encore écrite.")
        PENDING.add("CP5.5")
        RESULTS["CP5.5b"] = wb.check("CP5.5b", ..., quiet=True)
    elif not ok:
        prop("CP5.5", False, "", f"rnn_forward lève une erreur : {result}")
    else:
        try:
            H_all, h_last = pair("CP5.5", "rnn_forward", result, names=("H_all", "h_last"))
            want_all, want_last = oracle_5(X_S, H0_S, WX_S, WH_S, B_S)
            got_all, got_last = as_numpy(H_all), as_numpy(h_last)
            prop("CP5.5", got_all is not None and got_all.shape == (3, 5, 4) and got_last is not None
                 and got_last.shape == (3, 4),
                 "H_all a la forme (N, T, H) = (3, 5, 4) et h_last la forme (N, H) = (3, 4).",
                 f"attendu H_all de forme (3, 5, 4) et h_last de forme (3, 4), reçu {np.shape(H_all)}"
                 f"{' (une liste)' if isinstance(H_all, list) else ''} et "
                 f"{np.shape(h_last)} : H_all empile les T états après chaque pas, sur l'axe 1, sans h0.")
            if got_all is not None and got_all.shape == (3, 5, 4):
                gap = float(np.abs(got_all - want_all).max())
                prop("CP5.5", gap < 1e-10, "les états de chaque pas sont ceux de nn.RNN (mêmes poids, batch_first=True).",
                     f"l'écart à nn.RNN vaut {gap:.2e} : chaque pas reçoit X[:, t] (l'axe du temps est l'axe 1) et "
                     "l'état rendu par le pas précédent, en partant de h0.")
            if got_last is not None and got_last.shape == (3, 4):
                prop("CP5.5", np.abs(got_last - want_last).max() < 1e-10,
                     "h_last est l'état après le dernier pas (le h_n de nn.RNN).",
                     "h_last doit être l'état après le DERNIER pas, H_all[:, -1].")
            prop("CP5.5", np.array_equal(X_in, X_S), "rnn_forward ne modifie pas X.",
                 "rnn_forward modifie le X qu'elle reçoit : elle doit le laisser intact.")
            # other sizes, and a starting state that is not zero: are the arguments used?
            rng_o = np.random.default_rng(551)
            X_o, h0_o = rng_o.normal(size=(2, 3, 5)), rng_o.normal(size=(2, 6))
            Wx_o, Wh_o, b_o = rng_o.normal(0, 0.5, (5, 6)), rng_o.normal(0, 0.5, (6, 6)), rng_o.normal(0, 0.5, 6)
            ok_o, out_o = safe(rnn_forward, X_o, h0_o, Wx_o, Wh_o, b_o)
            want_o = oracle_5(X_o, h0_o, Wx_o, Wh_o, b_o)[0]
            try:
                same_o = bool(ok_o and np.allclose(as_numpy(out_o[0]), want_o, rtol=1e-10, atol=1e-12))
            except (TypeError, ValueError, IndexError):
                same_o = False
            prop("CP5.5", same_o, "avec d'autres tailles (N = 2, T = 3, D = 5, H = 6) et un h0 non nul, les états de nn.RNN.",
                 "avec N = 2, T = 3, D = 5, H = 6 et un h0 non nul, rnn_forward ne rend pas les états de nn.RNN : "
                 "utilise-t-elle h0 et les tailles de ses arguments (pas 5 pas ni 4 neurones écrits en dur) ?")
            ok_1, out_1 = safe(rnn_forward, X_o[:, :1], h0_o, Wx_o, Wh_o, b_o)
            try:
                one_ok = bool(ok_1 and as_numpy(out_1[0]).shape == (2, 1, 6)
                              and np.allclose(as_numpy(out_1[1]), want_o[:, 0], rtol=1e-10, atol=1e-12))
            except (TypeError, ValueError, IndexError):
                one_ok = False
            prop("CP5.5", one_ok, "une séquence d'un seul pas donne H_all de forme (N, 1, H).",
                 "avec T = 1, attendu H_all de forme (2, 1, 6) et h_last = rnn_cell(X[:, 0], h0, ...).")
            if got_all is not None and got_all.shape == (3, 5, 4):
                code_check("CP5.5b", got_all[1, 4, 0], "H_all[1, 4, 0] : séquence 1, pas 4 (le dernier), neurone 0.")
        except StopCheck:                                   # pair() printed why
            PROPERTIES["CP5.5"].append(("rnn_forward rend le couple (H_all, h_last)", False))

In [ ]:
# Checks of fit_early_stopping (CP5.6), against the contract written once more (about thirty seconds)
if exam_over() and ready("CP5.6", "torch", "nn", "make_cnn", "make_loaders", "fit_early_stopping"):
    PROPERTIES["CP5.6"] = []
    PENDING.discard("CP5.6")
    RESULTS.pop("CP5.6b", None)
    reference_fit = exams.cp5_fit_early_stopping        # the contract of CP5.6 a), the oracle of these checks

    def run(function, max_epochs, patience, lr, **sizes):
        """(ok, history or error, final weights, saved weights) of one training from a new CNN and new loaders."""
        path = Path(tempfile.mkdtemp()) / "best.pt"
        model = make_cnn()
        ok, history = safe(function, model, *make_loaders(**sizes), max_epochs, patience, path, lr=lr)
        try:
            saved = torch.load(path, weights_only=True) if ok and path.exists() else None
        except Exception:                                   # not a state_dict (a whole model, a dict of other things)
            saved = None
        return ok, history, {k: v.detach().clone() for k, v in model.state_dict().items()}, saved

    def same_weights(a, b):
        return (a is not None and b is not None and set(a) == set(b)
                and all(torch.allclose(a[k].double(), b[k].double(), rtol=1e-7, atol=1e-10) for k in b))

    def losses(history, key):
        """The list history[key] as floats, or None (missing key, not a list of numbers)."""
        try:
            values = as_numpy([float(v) for v in history[key]])
        except (TypeError, ValueError, KeyError):
            return None
        return values

    ok, mine, mine_final, mine_saved = run(fit_early_stopping, 30, 3, 3e-3)
    if ok is None:
        print("⏳ Ex CP5.6 : fit_early_stopping n'est pas encore écrite.")
        PENDING.add("CP5.6")
        RESULTS["CP5.6b"] = wb.check("CP5.6b", ..., quiet=True)
    elif not ok:
        prop("CP5.6", False, "", f"fit_early_stopping lève une erreur : {mine}")
    elif mine is None:
        prop("CP5.6", False, "", "fit_early_stopping renvoie None : as-tu oublié le return ?")
    elif not isinstance(mine, dict) or not {"train_loss", "val_loss", "best_epoch"} <= set(mine):
        prop("CP5.6", False, "", "fit_early_stopping doit rendre un dictionnaire aux clés \"train_loss\", \"val_loss\" et "
             f"\"best_epoch\" ; reçu {type(mine).__name__}" + (f" de clés {sorted(mine)}" if isinstance(mine, dict) else "") + ".")
    else:
        _, ref, ref_final, ref_saved = run(reference_fit, 30, 3, 3e-3)
        mine_tr, mine_va = losses(mine, "train_loss"), losses(mine, "val_loss")
        main_ok = False                                     # the losses of every epoch are those of the contract
        if mine_tr is None or mine_va is None:
            prop("CP5.6", False, "", "history[\"train_loss\"] et history[\"val_loss\"] doivent être des listes de nombres "
                 "(un float par epoch : loss.item()).")
        else:
            kinds = sorted({type(v).__name__ for key in ("train_loss", "val_loss") for v in mine[key]
                            if not isinstance(v, float)})
            prop("CP5.6", not kinds, "les losses rendues sont des float Python.",
                 f"history contient des {', '.join(kinds)} : range chaque loss comme un float Python (loss.item()), "
                 "pas comme un tenseur.")
            ref_tr, ref_va = as_numpy(ref["train_loss"]), as_numpy(ref["val_loss"])
            n = min(len(mine_tr), len(ref_tr), len(mine_va))
            bad_tr = next((k for k in range(n) if not np.isclose(mine_tr[k], ref_tr[k], rtol=1e-6, atol=0)), None)
            bad_va = next((k for k in range(n) if not np.isclose(mine_va[k], ref_va[k], rtol=1e-6, atol=0)), None)
            if len(mine_tr) != len(mine_va):
                prop("CP5.6", False, "", f"{len(mine_tr)} losses d'entraînement pour {len(mine_va)} de validation : une "
                     "valeur de chaque par epoch faite.")
            elif bad_tr is not None and (bad_va is None or bad_tr <= bad_va):
                prop("CP5.6", False, "", f"la loss d'entraînement de l'epoch {bad_tr + 1} vaut {mine_tr[bad_tr]:.6f}, "
                     f"attendu {ref_tr[bad_tr]:.6f} : un seul optimiseur Adam(lr=lr) pour tout l'entraînement, "
                     "model.train(), la cross-entropy moyenne sur les logits, et la moyenne PAR EXEMPLE des mini-batches "
                     "de l'epoch.")
            elif bad_va is not None:
                prop("CP5.6", False, "", f"la loss de validation de l'epoch {bad_va + 1} vaut {mine_va[bad_va]:.6f}, "
                     f"attendu {ref_va[bad_va]:.6f} : model.eval() avant la validation (sinon le dropout éteint des "
                     "neurones), torch.no_grad(), tout val_loader, et la moyenne PAR EXEMPLE (le dernier mini-batch a "
                     "124 images, pas 300).")
            else:
                main_ok = True
                prop("CP5.6", True, "à chaque epoch, les losses d'entraînement et de validation sont celles du contrat "
                     "(mode entraînement puis évaluation, moyennes par exemple).", "")
                prop("CP5.6", len(mine_va) == len(ref_va),
                     f"l'entraînement s'arrête après {len(ref_va)} epochs : 3 epochs de suite sans progrès.",
                     f"{len(mine_va)} epochs faites, attendu {len(ref_va)} : l'arrêt a lieu dès que patience = 3 epochs "
                     "DE SUITE n'ont pas fait strictement mieux que la meilleure loss de validation.")
            if main_ok and len(mine_va) != len(ref_va):
                print("   (best_epoch et les poids gardés ne se vérifient qu'une fois l'arrêt juste)")
            elif main_ok:
                prop("CP5.6", mine["best_epoch"] == ref["best_epoch"],
                     f"best_epoch = {ref['best_epoch']}, l'epoch de la plus petite loss de validation, comptée à partir de 1.",
                     f"best_epoch = {mine['best_epoch']!r}, attendu {ref['best_epoch']} : l'epoch (numérotée à partir de 1) "
                     "de la plus petite loss de validation.")
                prop("CP5.6", same_weights(mine_saved, ref_saved), "le fichier path contient le state_dict de la meilleure epoch.",
                     "le fichier path ne contient pas les poids de la meilleure epoch : torch.save(model.state_dict(), path) "
                     "à chaque progrès strict, et seulement alors.")
                prop("CP5.6", same_weights(mine_final, ref_final), "à la fin, le modèle a repris les poids de la meilleure epoch.",
                     "à la fin, le modèle n'a pas les poids de la meilleure epoch : model.load_state_dict(torch.load(path, "
                     "weights_only=True)) avant le return.")
            if bad_tr is not None or bad_va is not None or len(mine_tr) != len(mine_va):
                print("   (l'arrêt, best_epoch et les poids gardés ne se vérifient qu'une fois les losses de chaque epoch justes)")
        # a learning rate of 0: the validation loss never changes, so it never improves STRICTLY after epoch 1
        ok_0, flat, _, _ = run(fit_early_stopping, 6, 2, 0.0, n_train=256, n_val=128)
        try:
            n_flat, best_flat = len(flat["val_loss"]), flat["best_epoch"]
            moved = len({round(float(v), 12) for v in flat["val_loss"]}) > 1
        except (TypeError, KeyError, ValueError):
            n_flat, best_flat, moved = None, None, False
        if best_flat == 0:
            why = "les epochs se numérotent à partir de 1."
        elif moved:
            why = "la loss de validation a changé : l'optimiseur utilise-t-il le lr reçu, et la validation se fait-elle en mode évaluation (model.eval()) ?"
        elif isinstance(best_flat, (int, np.integer)) and best_flat > 1:
            why = "seul un progrès STRICT compte : une loss égale à la meilleure n'en est pas un."
        else:
            why = "l'arrêt vient dès que patience (le patience reçu) epochs de suite n'ont pas progressé."
        prop("CP5.6", ok_0 and n_flat == 3 and best_flat == 1,
             "avec lr = 0 (une loss de validation qui ne bouge plus), best_epoch = 1 et l'arrêt après 1 + patience epochs.",
             "avec lr = 0, patience = 2 et max_epochs = 6, la loss de validation est la même à chaque epoch : attendu "
             "best_epoch = 1 et 3 epochs faites" + (f", reçu {n_flat} epochs et best_epoch = {best_flat!r} : {why}"
                                                    if n_flat is not None else " : " + why))
        # max_epochs and lr come from the arguments
        ok_2, short, _, _ = run(fit_early_stopping, 2, 5, 1e-3, n_train=256, n_val=128)
        _, short_ref, _, _ = run(reference_fit, 2, 5, 1e-3, n_train=256, n_val=128)
        try:
            short_ok = bool(ok_2 and len(short["val_loss"]) == 2
                            and np.allclose([float(v) for v in short["val_loss"]], short_ref["val_loss"], rtol=1e-8, atol=0))
        except (TypeError, KeyError, ValueError):
            short_ok = False
        prop("CP5.6", short_ok, "avec max_epochs = 2 et lr = 1e-3 (256 images), les deux epochs du contrat.",
             "avec max_epochs = 2, patience = 5 et lr = 1e-3 sur 256 images, attendu les deux epochs du contrat : "
             + ("fit_early_stopping utilise-t-elle ses arguments (pas 30 epochs ni lr = 3e-3 écrits en dur) ?" if main_ok
                else "c'est la conséquence de l'écart des losses ci-dessus."))
        if mine_tr is not None and mine_va is not None and isinstance(mine.get("best_epoch"), (int, np.integer)) \
                and 1 <= mine["best_epoch"] <= len(mine_va):
            code_check("CP5.6b", [int(mine["best_epoch"]), float(mine_va[mine["best_epoch"] - 1])],
                       "best_epoch puis la loss de validation de cette epoch.")

### Les réponses de ta feuille

**CP5.1 — Forme de sortie ou erreur ? Six extraits PyTorch**

In [ ]:
# CP5.1: the values written on your answer sheet
answer_CP5_1a = ...  # a) a shape like "(1, 2)", or "erreur"
answer_CP5_1b = ...  # b) a shape like "(1, 2)", or "erreur"
answer_CP5_1c = ...  # c) a shape like "(1, 2)", or "erreur"
answer_CP5_1d = ...  # d) a shape like "(1, 2)", or "erreur"
answer_CP5_1e = ...  # e) a shape like "(1, 2)", or "erreur"
answer_CP5_1f = ...  # f) a shape like "(1, 2)", or "erreur"

if exam_over():
    for letter, answer in zip("abcdef", [as_outcome(answer_CP5_1a), as_outcome(answer_CP5_1b), as_outcome(answer_CP5_1c), as_outcome(answer_CP5_1d), as_outcome(answer_CP5_1e), as_outcome(answer_CP5_1f)]):
        RESULTS[f"CP5.1{letter}"] = wb.check(f"CP5.1{letter}", answer)

**CP5.2 — Formes, paramètres et champ récepteur d'un petit CNN**

In [ ]:
# CP5.2: the values written on your answer sheet
answer_CP5_2a = ...  # a) a shape like "(1, 2, 3, 4)"
answer_CP5_2b = ...  # b) a shape like "(1, 2, 3, 4)"
answer_CP5_2c = ...  # c) a shape like "(1, 2, 3, 4)"
answer_CP5_2d = ...  # d) a list of five numbers
answer_CP5_2e = ...  # e) a number of multiply-adds
answer_CP5_2f = ...  # f) a size in pixels

if exam_over():
    for letter, answer in zip("abcdef", [shape_text(answer_CP5_2a), shape_text(answer_CP5_2b), shape_text(answer_CP5_2c), answer_CP5_2d, answer_CP5_2e, answer_CP5_2f]):
        RESULTS[f"CP5.2{letter}"] = wb.check(f"CP5.2{letter}", answer)

**CP5.3 — Un pas de LSTM à la main**

In [ ]:
# CP5.3: the values written on your answer sheet
answer_CP5_3a = ...  # a) [i, f, g, o], 4 decimals
answer_CP5_3b = ...  # b) c1, 4 decimals
answer_CP5_3c = ...  # c) h1, 4 decimals
answer_CP5_3d = ...  # d) number of parameters

if exam_over():
    for letter, answer in zip("abcd", [answer_CP5_3a, answer_CP5_3b, answer_CP5_3c, answer_CP5_3d]):
        RESULTS[f"CP5.3{letter}"] = wb.check(f"CP5.3{letter}", answer)

**CP5.4 — Les lettres des quatre lignes fautives (a)**

In [ ]:
# CP5.4: the values written on your answer sheet
answer_CP5_4a = ...  # a) four letters, for example "XYZW"

if exam_over():
    for letter, answer in zip("a", [sorted_letters(answer_CP5_4a)]):
        RESULTS[f"CP5.4{letter}"] = wb.check(f"CP5.4{letter}", answer)

**CP5.7 — Diagnostiquer quatre paires de courbes**

In [ ]:
# CP5.7: the values written on your answer sheet
answer_CP5_7a = ...  # a) a letter A to F
answer_CP5_7b = ...  # b) a letter A to F
answer_CP5_7c = ...  # c) a letter A to F
answer_CP5_7d = ...  # d) a letter A to F

if exam_over():
    for letter, answer in zip("abcd", [letters_text(answer_CP5_7a), letters_text(answer_CP5_7b), letters_text(answer_CP5_7c), letters_text(answer_CP5_7d)]):
        RESULTS[f"CP5.7{letter}"] = wb.check(f"CP5.7{letter}", answer)

**CP5.8 — Fermi : un générateur de texte LSTM**

In [ ]:
# CP5.8: the values written on your answer sheet
answer_CP5_8a = ...  # a) parameters in millions, 1 decimal
answer_CP5_8b = ...  # b) a percentage, rounded to the unit
answer_CP5_8c = ...  # c) FLOP in units of 10**13, 1 decimal
answer_CP5_8d = ...  # d) seconds, rounded to the unit

if exam_over():
    for letter, answer in zip("abcd", [answer_CP5_8a, answer_CP5_8b, answer_CP5_8c, answer_CP5_8d]):
        RESULTS[f"CP5.8{letter}"] = wb.check(f"CP5.8{letter}", answer)

**CP5.10 — Retour sur les parties III et IV**

In [ ]:
# CP5.10: the values written on your answer sheet
answer_CP5_10a = ...  # a) [mean, std] of the scaler, 4 decimals
answer_CP5_10b = ...  # b) z-score, 3 decimals
answer_CP5_10c = ...  # c) number of fits of the scaler
answer_CP5_10d = ...  # d) dW as a list of two rows
answer_CP5_10e = ...  # e) db as a list
answer_CP5_10f = ...  # f) dX as a list of two rows

if exam_over():
    for letter, answer in zip("abcdef", [answer_CP5_10a, answer_CP5_10b, answer_CP5_10c, answer_CP5_10d, answer_CP5_10e, answer_CP5_10f]):
        RESULTS[f"CP5.10{letter}"] = wb.check(f"CP5.10{letter}", answer)

**CP5.11 — Questions flash : paramètres et tailles**

In [ ]:
# CP5.11: the values written on your answer sheet
answer_CP5_11a = ...  # a) number of parameters
answer_CP5_11b = ...  # b) number of parameters
answer_CP5_11c = ...  # c) the output height
answer_CP5_11d = ...  # d) number of parameters

if exam_over():
    for letter, answer in zip("abcd", [answer_CP5_11a, answer_CP5_11b, answer_CP5_11c, answer_CP5_11d]):
        RESULTS[f"CP5.11{letter}"] = wb.check(f"CP5.11{letter}", answer)

**CP5.12 — Rétropropager dans une convolution 1D**

In [ ]:
# CP5.12: the values written on your answer sheet
answer_CP5_12a = ...  # a) [y1, y2, y3]
answer_CP5_12c = ...  # c) [dL/dw1, dL/dw2, dL/db]
answer_CP5_12d = ...  # d) [dL/dx1, ..., dL/dx4]

if exam_over():
    for letter, answer in zip("acd", [answer_CP5_12a, answer_CP5_12c, answer_CP5_12d]):
        RESULTS[f"CP5.12{letter}"] = wb.check(f"CP5.12{letter}", answer)

### Bilan

In [ ]:
# Summary of the automatic checks
import re


def natural(sub_id):
    """CP5.2a before CP5.10a: the question number as a number, then the letter."""
    number, letter = re.match(r"CP5\.(\d+)(.*)", sub_id).groups()
    return int(number), letter


if exam_over():
    done = {k: r for k, r in RESULTS.items() if r.status != "pending"}
    right = sorted((k for k, r in done.items() if r), key=natural)
    wrong = sorted((k for k, r in done.items() if not r), key=natural)
    pending = sorted((k for k, r in RESULTS.items() if r.status == "pending"), key=natural)
    print(f"Réponses vérifiées : {len(right)} juste(s) sur {len(done)} ; pas remplies : {len(pending)}.")
    if wrong:
        print("À revoir avec le corrigé :", ", ".join(wrong))
    for question in ("CP5.5", "CP5.6"):
        checked_props = [ok for _, ok in PROPERTIES.get(question, [])]
        state = " (pas encore écrite : ⏳)" if question in PENDING else ""
        print(f"Propriétés du code de {question} : {sum(checked_props)} sur {len(checked_props)}{state}.")
    print("Note ensuite ta copie avec le barème de 03_examen_corrige.md : il donne aussi les points des démarches, "
          "des justifications et des questions rédigées, que cette vérification ne voit pas.")

**Ce que cette vérification ne voit pas** : les démarches et les justifications ; les explications (CP5.1 g, CP5.2 g, CP5.3 e, CP5.4 b, CP5.9, CP5.12 b, CP5.13, CP5.14) ; et, pour CP5.5 et CP5.6, les nombres recopiés sur ta feuille (seules tes fonctions sont vérifiées). Tout cela se corrige avec `03_examen_corrige.md`. Ensuite : la remédiation du corrigé (relis la synthèse, `05_synthese.md`, pour les notions ratées), puis le mini-projet.